In [0]:
%run ../transform/operaciones_replica

In [0]:
import logging
from pyspark.sql.types import DateType, DecimalType
from pyspark.sql import functions as F

# Inicialización de parámetros lógicos mediante Widgets para mantener la simetría de orquestación
dbutils.widgets.text("full_load", "0")
full_load = int(dbutils.widgets.get("full_load"))
carga_full = (full_load == 1)

# Configuración del motor de logs estándar para auditoría en producción
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("dim_instrumento")

In [0]:
ruta_cotizaciones = "/Volumes/platform_dev/iol_challenge/yfinance/cotizaciones_yahoo.csv"
tabla_destino = "platform_dev.gold_byma.dim_instrumento"

try:
    # Programación Defensiva: Si el catálogo analítico Gold sufre un reset, el script
    # auto-detecta la ausencia de la tabla e invoca de forma transparente al DDL centralizado.
    if not spark.catalog.tableExists(tabla_destino):
        logger.warning(f"La tabla {tabla_destino} no existe. Iniciando creación...")
        dbutils.notebook.run("../DDL/creacion_tablas", 0)
        carga_full = True # Activación forzada de carga masiva histórica
    elif carga_full:
        logger.warning(f"Carga full forzada manualmente para {tabla_destino}.")
    else:
        logger.info(f"La tabla {tabla_destino} existe.")
except Exception:
    logger.exception(f"Error al validar la tabla {tabla_destino}")
    raise

In [0]:
# Desacople de APIs Externas: Leemos las cotizaciones de Yahoo Finance desde el volumen intermedio.
# Esto aísla el pipeline principal de potenciales caídas de red o límites de cuota (Rate Limits) de la API externa.
df_cotizaciones = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(ruta_cotizaciones)
)

# Fortificación y Precisión del Dato
df_cotizaciones_silver = (
    df_cotizaciones
    .withColumn("fecha", F.col("fecha").cast(DateType()))
    .withColumn("fuente_api", F.lit("yfinance"))
    .withColumn("precio_apertura", F.lit(None).cast(DecimalType(18, 4)))
    # IMPORTANTE: Casteamos los precios de cierre estrictamente a DecimalType(18,4).
    # El uso de floats nativos introduciría ruidos de redondeo destructivos al calcular desvíos porcentuales.
    .withColumn("precio_cierre", F.col("precio_cierre").cast(DecimalType(18, 4)))
)

# Persistencia en Silver: Guardamos las cotizaciones normalizadas listas para el consumo de Hechos
(
    df_cotizaciones_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("platform_dev.silver_byma.cotizaciones_historicas")
)

In [0]:
# Extraemos el universo de activos únicos mapeados desde las transacciones liquidadas
df_dim_instrumento = (
    spark.table("platform_dev.silver_byma.operaciones_replica")
    .select(
        F.col("simbolo_titulo").alias("simbolo"),
        F.col("descripcion_titulo").alias("descripcion"),
        F.col("moneda").alias("moneda_base")
    )
    # Corregido: Filtramos por "simbolo" porque el alias ya cambió el nombre de la columna
    .filter(F.col("simbolo").isNotNull())
    .dropDuplicates(["simbolo"])
    
    # MOTOR SEMÁNTICO: Clasificación de instrumentos mediante reglas duras de negocio.
    # Traduce textos libres e inconsistentes de BYMA en clases analíticas homogéneas para el BI.
    .withColumn(
        "tipo_instrumento",
        F.when(
            F.lower(F.col("descripcion")).contains("bono") | 
            F.lower(F.col("descripcion")).contains("tesoro") | 
            F.lower(F.col("descripcion")).contains("bopreal") |
            F.col("simbolo").isin("AL30", "AL30D", "TZX28", "TTM26", "AN29", "BPY26"), 
            "Bono soberano"
        )
        .when(
            F.col("simbolo").isin("YPFD", "PAMP", "ALUA", "COME", "TXAR", "CGPA2", "AGRO", "CELU", "SEMI") |
            F.lower(F.col("descripcion")).contains("camuzzi") |
            F.lower(F.col("descripcion")).contains("agrometal") |
            F.lower(F.col("descripcion")).contains("celulosa") |
            F.lower(F.col("descripcion")).contains("semino") |
            F.col("simbolo").startswith("GFG"), 
            "Accion local"
        )
        .otherwise("Cedear") # Fallback por defecto para la renta variable extranjera (Cedears)
    )
    
    # Enriquecimiento de Tickers: Formateamos el sufijo bursátil oficial (.BA) para las acciones locales
    # garantizando la compatibilidad exacta con los diccionarios analíticos de mercado.
    .withColumn(
        "ticker_api",
        F.when(F.col("tipo_instrumento") == "Accion local", F.concat(F.col("simbolo"), F.lit(".BA")))
         .otherwise(F.col("simbolo"))
    )
    .withColumn(
        "_created_at",
        F.current_timestamp()
    )
)

# Publicación como vista temporal en el catálogo de memoria de Spark
df_dim_instrumento.createOrReplaceTempView("dim_instrumento")

In [0]:
try:
    if carga_full:
        logger.warning(f"Carga full: Reconstruyendo tabla atómicamente {tabla_destino}")
        # DETALLE DE ARQUITECTURA CRÍTICO: Declaramos explícitamente las columnas funcionales omitiendo 'instrumento_sk'.
        # Esto le permite al comando 'INSERT OVERWRITE' de Delta Lake limpiar los datos antiguos pero CONSERVAR la estructura
        # del DDL intacta, haciendo que el contador de la secuencia IDENTITY se reinicie de forma limpia desde 1.
        spark.sql(f"""
            INSERT OVERWRITE {tabla_destino} (simbolo, descripcion, tipo_instrumento, moneda_base, ticker_api, _created_at)
            SELECT simbolo, descripcion, tipo_instrumento, moneda_base, ticker_api, current_timestamp() 
            FROM dim_instrumento
        """)
        logger.info("Estructura e IDs reseteados correctamente.")
    else:
        logger.info(f"Carga incremental: Aplicando MERGE defensivo en {tabla_destino} para proteger el histórico.")
        # ESTRATEGIA SCD TIPO 1 DEFENSIVA: Al utilizar un MERGE condicional basado en la Clave Natural ('simbolo'), 
        # logramos actualizar descripciones si cambiaron en origen, pero mantenemos completamente inmutable la clave 'instrumento_sk'.
        # Esto blinda las relaciones históricas del pasado y evita que la tabla de hechos incremental rompa sus joins analíticos.
        spark.sql(f"""
            MERGE INTO {tabla_destino} AS target
            USING dim_instrumento AS source
            ON target.simbolo = source.simbolo
            WHEN MATCHED AND (
                target.descripcion <> source.descripcion OR 
                target.tipo_instrumento <> source.tipo_instrumento OR 
                target.moneda_base <> source.moneda_base OR 
                target.ticker_api <> source.ticker_api
            ) THEN
              UPDATE SET 
                target.descripcion = source.descripcion,
                target.tipo_instrumento = source.tipo_instrumento,
                target.moneda_base = source.moneda_base,
                target.ticker_api = source.ticker_api
            WHEN NOT MATCHED THEN
              INSERT (simbolo, descripcion, tipo_instrumento, moneda_base, ticker_api, _created_at)
              VALUES (source.simbolo, source.descripcion, source.tipo_instrumento, source.moneda_base, source.ticker_api, current_timestamp())
        """)
        logger.info("MERGE completado con éxito. Claves SK preservadas.")
except Exception:
    logger.exception(f"Error crítico en la persistencia de {tabla_destino}")
    raise